# 09 — Apply Registered Missed Delivery Model

## Objective

This notebook applies the registered Missed Delivery model to the daily feature dataset generated by the feature-engineering pipeline.

The workflow:

- reads the prepared scoring feature table,
- loads Version 2 of the registered Fabric ML model,
- performs batch inference,
- preserves the business metadata associated with each prediction,
- prepares the final prediction dataset for downstream reporting and monitoring.

In [ ]:
%pip install --upgrade --force-reinstall \
scikit-learn==1.9.0 \
pandas==3.0.5 \
numpy==2.5.1 \
xgboost==3.4.1 \
joblib==1.5.3

In [ ]:
import notebookutils

print("Dependencies installed. Restarting Python...")

notebookutils.session.restartPython()

In [ ]:
import sys
import pandas as pd
import numpy as np
import sklearn
import xgboost
import joblib
import mlflow

print("Python:", sys.version)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgboost.__version__)
print("joblib:", joblib.__version__)
print("mlflow:", mlflow.__version__)

assert sys.version_info[:2] == (3, 12)
assert pd.__version__ == "3.0.5"
assert np.__version__ == "2.5.1"
assert sklearn.__version__ == "1.9.0"
assert xgboost.__version__ == "3.4.1"
assert joblib.__version__ == "1.5.3"

print("Frozen inference runtime: PASS")

In [ ]:
model = mlflow.pyfunc.load_model(
    "models:/md_early_warning/2"
)

In [ ]:
import notebookutils

with notebookutils.data.connect_to_artifact(
    "BRZ_SLV_Lakegistica",
    artifact_type="Lakehouse"
) as conn:
    scoring_pd = conn.query("""
        SELECT *
        FROM md_scoring_features
    """)

In [ ]:
print(scoring_pd.shape)

In [ ]:
import json

with open(
    "./builtin/feature_schema.json",
    "r",
    encoding="utf-8"
) as f:
    feature_schema = json.load(f)

expected_features = feature_schema["expected_features"]

X_model = scoring_pd[expected_features].copy()

print("Model input shape:", X_model.shape)

In [ ]:
import numpy as np
import pandas as pd

categorical_features = [
    "Grupo_raiz",
    "customer",
    "uat",
    "destination"
]

numeric_features = [
    c for c in expected_features
    if c not in categorical_features
]

X_model = scoring_pd[expected_features].copy()

# Categorical features: pandas object dtype for MLflow compatibility
for c in categorical_features:
    X_model[c] = X_model[c].astype("object")

# Numerical features: double
for c in numeric_features:
    X_model[c] = pd.to_numeric(
        X_model[c],
        errors="coerce"
    ).astype("float64")

print(X_model.shape)
print(X_model[categorical_features].dtypes)

In [ ]:
print(X_model.shape)

print(
    X_model[
        [
            "history_available_D7",
            "available_lag_count",
            "history_records_before_D",
            "month",
            "day_of_week",
            "week_of_year"
        ]
    ].dtypes
)

In [ ]:
print(X_model[categorical_features].dtypes)

print(
    X_model[
        [
            "available_lag_count",
            "month",
            "day_of_week",
            "week_of_year"
        ]
    ].dtypes
)

In [ ]:
predictions_pd = model.predict(X_model)

display(predictions_pd)

In [ ]:
result_pd = pd.concat(
    [
        scoring_pd[
            [
                "scoring_date",
                "target_date",
                "request",
                "Grupo_raiz",
                "customer",
                "uat",
                "destination"
            ]
        ].reset_index(drop=True),
        predictions_pd.reset_index(drop=True)
    ],
    axis=1
)

display(result_pd)

In [ ]:
print(result_pd.shape)
print(result_pd.columns.tolist())
print(result_pd["alert_flag"].value_counts(dropna=False))

In [ ]:
from datetime import datetime, timezone
import uuid

MODEL_NAME = "md_early_warning"
MODEL_VERSION = "2"

run_id = str(uuid.uuid4())
run_timestamp_utc = datetime.now(timezone.utc)

result_pd["model_name"] = MODEL_NAME
result_pd["model_version"] = MODEL_VERSION
result_pd["run_id"] = run_id
result_pd["run_timestamp_utc"] = run_timestamp_utc

print(result_pd.shape)
print(result_pd.columns.tolist())

In [ ]:
#assert result_pd.shape == (99, 15)
assert result_pd["run_id"].notna().all()
assert result_pd["model_version"].eq("2").all()
assert result_pd["threshold"].eq(0.08).all()

print("Prediction output validation: PASS")

In [ ]:
from deltalake import write_deltalake
import notebookutils

ONELAKE_WORKSPACE_ID = "<ONELAKE_WORKSPACE_ID>"
LAKEHOUSE_ID = "<LAKEHOUSE_ID>"

target_path = (
    f"abfss://{ONELAKE_WORKSPACE_ID}"
    "@onelake.dfs.fabric.microsoft.com/"
    f"{LAKEHOUSE_ID}/Tables/dbo/md_predictions"
)

token = notebookutils.credentials.getToken("storage")

scoring_date_value = str(result_pd["scoring_date"].iloc[0])

write_deltalake(
    target_path,
    result_pd,
    mode="overwrite",
    predicate=f"scoring_date = '{scoring_date_value}'",
    storage_options={
        "bearer_token": token,
        "use_fabric_endpoint": "true"
    }
)

print(
    f"md_predictions saved successfully for scoring_date={scoring_date_value}"
)

In [ ]:
table_path = target_path

import duckdb
display(duckdb.sql(f"select * from delta_scan('{table_path}') limit 1000 ").df())